# Week 3 — LMS Adaptive Filtering

We now move from fixed filtering to adaptive noise cancellation. Channel 2 contains a 1 kHz desired signal plus a 10 kHz interference. A correlated 10 kHz reference is supplied to an LMS filter, which learns an estimate of the interference.

**Goal:** compare learning rates μ = 0.001, 0.01 and 0.1 using the same data and measure SNR, MSE and convergence behavior.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dsp.signal_generator import AcquisitionConfig, generate_channels
from dsp.adaptive_filters import LMSFilter, mse, snr_db

## 1. Build the adaptive-filter test case

In [ ]:
cfg = AcquisitionConfig()
t, clean, noisy, metadata = generate_channels(cfg)

# Channel 2 is the deliberate adaptive-cancellation test case.
desired_clean = clean[:, 1]
measured = noisy[:, 1]

# Reference sensor: correlated with the 10 kHz interference, with a small
# phase offset to make the experiment less idealized than a copy of d.
reference = np.sin(2 * np.pi * 10_000 * t + np.pi / 12)

print('Test channel      : CH2')
print('Sampling rate     :', f'{cfg.fs/1000:.1f} kHz')
print('Desired component : 1 kHz')
print('Interference      : 10 kHz')
print('LMS order         : 32')
print('Reference RMS     :', f'{np.sqrt(np.mean(reference**2)):.4f} V')

## 2. Baseline signal

In [ ]:
def spectrum(x, fs):
    n = len(x)
    f = np.fft.rfftfreq(n, 1 / fs)
    a = np.abs(np.fft.rfft(x)) / n
    return f, a

baseline_snr = snr_db(desired_clean, measured)
print(f'Baseline SNR: {baseline_snr:.2f} dB')

f, mag = spectrum(measured, cfg.fs)
plt.figure(figsize=(11, 4))
plt.plot(f / 1000, 20 * np.log10(np.maximum(mag, 1e-12)))
plt.xlim(0, 15)
plt.xlabel('Frequency [kHz]')
plt.ylabel('Magnitude [dB]')
plt.title('CH2 before adaptive cancellation')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Compare LMS learning rates

In [ ]:
learning_rates = [0.001, 0.01, 0.1]
n_order = 32
results = {}

for mu in learning_rates:
    filt = LMSFilter(order=n_order, learning_rate=mu)
    estimated_interference, error, weights = filt.adapt(reference, measured)
    results[mu] = {'estimate': estimated_interference, 'error': error, 'weights': weights}

    warmup = int(0.010 * cfg.fs)
    result_snr = snr_db(desired_clean, error, start=warmup)
    result_mse = mse(error, start=warmup)
    results[mu]['snr'] = result_snr
    results[mu]['mse'] = result_mse
    print(f'mu={mu:0.3f} | SNR={result_snr:7.2f} dB | MSE={result_mse:10.6f} V^2')

## 4. Learning curves

In [ ]:
window = 100
for mu in learning_rates:
    err = results[mu]['error']
    power = np.convolve(err**2, np.ones(window) / window, mode='valid')
    plt.figure(figsize=(11, 4))
    plt.plot(np.arange(len(power)) / cfg.fs * 1000, 10 * np.log10(np.maximum(power, 1e-14)))
    plt.xlabel('Time [ms]')
    plt.ylabel('Smoothed error power [dB]')
    plt.title(f'LMS learning curve, mu={mu}')
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 5. Before/after time-domain comparison

In [ ]:
best_mu = max(learning_rates, key=lambda x: results[x]['snr'])
filtered = results[best_mu]['error']
window = t < 0.008

plt.figure(figsize=(12, 5))
plt.plot(t[window] * 1000, measured[window], label='Measured CH2')
plt.plot(t[window] * 1000, filtered[window], label=f'LMS output (mu={best_mu})')
plt.plot(t[window] * 1000, desired_clean[window], label='Clean reference', linewidth=1.2)
plt.xlabel('Time [ms]')
plt.ylabel('Voltage [V]')
plt.title('Adaptive noise cancellation — time domain')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best measured learning rate: mu={best_mu}')
print(f'SNR improvement: {results[best_mu]["snr"] - baseline_snr:.2f} dB')

## 6. Frequency-domain comparison

In [ ]:
f1, a1 = spectrum(measured, cfg.fs)
f2, a2 = spectrum(filtered, cfg.fs)
plt.figure(figsize=(11, 4))
plt.plot(f1 / 1000, 20 * np.log10(np.maximum(a1, 1e-12)), label='Before LMS')
plt.plot(f2 / 1000, 20 * np.log10(np.maximum(a2, 1e-12)), label='After LMS')
plt.xlim(0, 15)
plt.xlabel('Frequency [kHz]')
plt.ylabel('Magnitude [dB]')
plt.title(f'CH2 spectrum before vs after LMS (mu={best_mu})')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Coefficient convergence

In [ ]:
weights = results[best_mu]['weights']
plt.figure(figsize=(11, 5))
for k in range(min(6, n_order)):
    plt.plot(t * 1000, weights[:, k], label=f'w[{k}]')
plt.xlabel('Time [ms]')
plt.ylabel('Coefficient value')
plt.title(f'First LMS coefficients, mu={best_mu}')
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Interpretation

The three learning rates deliberately show the speed/stability trade-off. A larger μ can adapt faster but produces greater misadjustment and can become unstable for sufficiently large step sizes. The measured values above are the experiment's results; they should be used in the project report rather than assumed target values.